# Simplicits Debug on Surgical Tissue Point Cloud (Gravity Only)

This notebook loads a **surface point cloud** from your pickle file and runs a **first-pass Simplicits simulation under gravity** (no tool constraints yet).

**Notes**
- This follows the structure of Kaolin's Simplicits Easy API example.
- If you run in **VS Code** and the Kaolin ipywidget visualizer fails, use the **k3d** visualization cells (they work in most environments).


In [1]:
# --- Imports ---
import os
import pickle
import numpy as np
import torch

# in notebook / script BEFORE importing kaolin
import sys
sys.path.insert(0, "/home/nan/Desktop/kaolin")
import kaolin as kal
# For point cloud visualization (recommended for VS Code + Jupyter)
#   pip install k3d
import k3d

print(kal.__file__)
# To get just the directory:
print(os.path.dirname(os.path.abspath(kal.__file__)))

# --- Use patched Sim2Real training loop ---
import sys
sys.path.insert(0, '/mnt/data')
import kaolin.physics.simplicits.easy_api_sim2real_UPDATED_anchor_boundary_fastckpt_v3 as ez


/home/nan/Desktop/kaolin/kaolin/__init__.py
/home/nan/Desktop/kaolin/kaolin


## 1) Load tissue point cloud from PKL

In [2]:
pkl_file_path = "/home/nan/Desktop/datasets/xpbd/StereoMIS_tissue_tool_trajectories_XPBD/sim_particles/tissue_pts_dnsampled_once.pkl"

with open(pkl_file_path, "rb") as f:
    real_seq = pickle.load(f)

print(type(real_seq), len(real_seq))
print("keys:", real_seq[0].keys())
print("xyz:", np.asarray(real_seq[0]["xyz"]).shape)

<class 'list'> 200
keys: dict_keys(['frame_id', 'xyz', 'rgb', 'opacity', 'indices'])
xyz: (38426, 3)


## 2) Choose a rest frame and move to GPU

In [3]:
# Choose which frame to treat as rest state
rest_frame_idx = 0

xyz = np.asarray(real_seq[rest_frame_idx]["xyz"], dtype=np.float32)  # (N,3)
N = xyz.shape[0]
print("N =", N)

# RAW points (keep these!)
pts_raw = torch.from_numpy(xyz).cuda()

# ---- explicit center/scale (save these for tool normalization!) ----
mn = pts_raw.min(dim=0).values
mx = pts_raw.max(dim=0).values
center_raw = 0.5 * (mn + mx)
half_extent = 0.5 * (mx - mn)
scale_raw = float(half_extent.max().detach().cpu())  # scalar

# Normalize tissue into roughly [-1,1]
pts = (pts_raw - center_raw) / max(scale_raw, 1e-8)

orig_pts = pts.clone()

print("tissue raw bbox min/max:",
      mn.detach().cpu().numpy(),
      mx.detach().cpu().numpy())
print("center_raw:", center_raw.detach().cpu().numpy(), "scale_raw:", scale_raw)
print("tissue normed bbox min/max:",
      pts.min(dim=0).values.detach().cpu().numpy(),
      pts.max(dim=0).values.detach().cpu().numpy())

N = 38426
tissue raw bbox min/max: [-0.76335067 -0.65025973  0.77168   ] [0.5815084  0.41067317 1.8512661 ]
center_raw: [-0.09092113 -0.11979328  1.3114731 ] scale_raw: 0.6724295616149902
tissue normed bbox min/max: [-1.        -0.7888803 -0.8027504] [1.        0.7888803 0.8027503]


In [4]:
# --- Normalize ALL real frames into the same coordinate frame as pts ---
real_xyz_norm = []
real_frame_ids = []
for fr in real_seq:
    real_frame_ids.append(int(fr["frame_id"]))
    xyz_np = np.asarray(fr["xyz"], dtype=np.float32)
    xyz_t = torch.from_numpy(xyz_np).to(device=pts.device)

    # Do the same normalizations to real points, compared to sim points
    xyz_norm = (xyz_t - center_raw) / max(scale_raw, 1e-8)
    real_xyz_norm.append(xyz_norm)

print("Loaded real frames:", real_frame_ids[0], "->", real_frame_ids[-1])
print("Example normalized frame shape:", real_xyz_norm[0].shape)

# Build dict for quick lookup
real_frame_to_xyz = {fid: xyz for fid, xyz in zip(real_frame_ids, real_xyz_norm)}
# frame_ids = real_frame_ids


Loaded real frames: 0 -> 199
Example normalized frame shape: torch.Size([38426, 3])


## 3) Quick visualization of rest point cloud (k3d)

In [5]:
plot = k3d.plot()
k3d_pts = k3d.points(orig_pts.detach().cpu().numpy(), point_size=0.01)
plot += k3d_pts
plot.display()

Output()

## 4) Material fields (constant for now)

In [6]:
# These are *per-point* material fields used by the elastic loss / simulation.
# Start simple: constant fields.
# Units are not super important for this debug step; tune later.

yms  = torch.full((N,), 2e5, device=pts.device, dtype=pts.dtype)   # Young's modulus
prs  = torch.full((N,), 0.45, device=pts.device, dtype=pts.dtype)  # Poisson ratio
rhos = torch.full((N,), 1000., device=pts.device, dtype=pts.dtype) # Density

# Approx volume: for surface point clouds this is not "true volume".
# Use a rough proxy based on bounding box volume to get reasonable scaling.
mn = pts.min(dim=0).values
mx = pts.max(dim=0).values
bbox_vol = float(torch.prod(mx - mn).detach().cpu())
approx_volume = max(bbox_vol, 1e-6)

print("approx_volume (bbox proxy):", approx_volume)

approx_volume (bbox proxy): 5.06619119644165


## 5) Build a scene first and train a SimplicitsObject (learn weights)

This step learns the **skinning weight field** \(w(x)\) (self-supervised) using elastic energy under random handle transforms.

Start with low iterations to debug the pipeline; increase later.


In [7]:
# load tool trajectory
import pickle
import numpy as np
import torch

tool_pkl_path = "/home/nan/Desktop/datasets/xpbd/StereoMIS_tissue_tool_trajectories_XPBD/sim_particles/tool_3d_poses.pkl"
with open(tool_pkl_path, "rb") as f:
    tool_data = pickle.load(f)

frame_ids = sorted(tool_data.keys())
print("tool frames:", frame_ids[0], "->", frame_ids[-1], "count:", len(frame_ids))

tool_t_raw = np.stack([np.asarray(tool_data[k]["t"], dtype=np.float32) for k in frame_ids], axis=0)  # (T,3)
tool_R_raw = np.stack([np.asarray(tool_data[k]["R"], dtype=np.float32) for k in frame_ids], axis=0)  # (T,3,3)

tool_t_raw_t = torch.from_numpy(tool_t_raw).to(device=pts_raw.device, dtype=pts_raw.dtype)
tool_R = torch.from_numpy(tool_R_raw).to(device=pts_raw.device, dtype=pts_raw.dtype)

# IMPORTANT: use RAW tissue normalization params, not orig_pts
# You must have computed these when loading tissue:
# center_raw, scale_raw from pts_raw (raw tissue points)
tool_t = (tool_t_raw_t - center_raw) / max(scale_raw, 1e-8)

print("tool_t range (normed):",
      tool_t.min(dim=0).values.detach().cpu().numpy(),
      tool_t.max(dim=0).values.detach().cpu().numpy())

# # sanity: distance from tool to tissue in normalized space
# internal0 = scene.get_object_deformed_pts(obj_idx)
# dmin = torch.norm(internal0 - tool_t[0][None, :], dim=1).min()
# print("min dist tool->tissue (internal) at t0:", float(dmin.detach().cpu()))

# --- Align real frames to tool frames (we only supervise frames that have tool pose) ---
aligned_frame_ids = [fid for fid in frame_ids if fid in real_frame_to_xyz]
if len(aligned_frame_ids) != len(frame_ids):
    missing = [fid for fid in frame_ids if fid not in real_frame_to_xyz]
    print("WARNING: missing real frames for some tool frames, e.g.:", missing[:10], "count:", len(missing))

# Rebuild aligned sequences
tool_t = tool_t[[frame_ids.index(fid) for fid in aligned_frame_ids]]
tool_R = tool_R[[frame_ids.index(fid) for fid in aligned_frame_ids]]
frame_ids = aligned_frame_ids
real_xyz_norm_aligned = [real_frame_to_xyz[fid] for fid in frame_ids]

print("Aligned frames:", frame_ids[0], "->", frame_ids[-1], "count:", len(frame_ids))


tool frames: 75 -> 199 count: 125
tool_t range (normed): [ 0.19295588 -1.1589513  -0.33955088] [ 0.8074347  -0.8500399   0.08211552]
Aligned frames: 75 -> 199 count: 125


## 6) Train two skinning-weight models (baseline vs Sim2Real)

This trains:
- **Baseline**: physical losses only (`ls=0`)
- **Sim2Real**: physical + Chamfer-to-real rollout (`ls>0`) using early frames.

You can resume from checkpoints in each run directory.

In [8]:

import os, json, time
from datetime import datetime

# --------------------------
# Train/Test split (sequence indices, not original frame ids)
# --------------------------
T_total = len(frame_ids)
train_T = min(60, T_total)          # use first train_T frames for Sim2Real supervision
eval_T0 = train_T
eval_T1 = T_total                   # evaluate on later frames by default (can shorten)

print(f"T_total={T_total}  train=[0,{train_T})  eval=[{eval_T0},{eval_T1})")

# --------------------------
# Training configs
# --------------------------
num_handles = 5

# Common training knobs (baseline + s2r)
common_train_kwargs = dict(
    num_handles=num_handles,
    training_num_steps=3000,
    training_lr_start=1e-3,
    training_lr_end=1e-3,
    training_le_coeff=1e-1,
    training_lo_coeff=1e6,
    training_log_every=50,
    normalize_for_training=True,
    # speed / ckpt knobs (supported by *_fastckpt.py)
    training_batch_size=10,
    num_samples=800,
    training_save_every=200,              # checkpoint cadence (steps)
    training_eval_every=200,              # quick eval cadence (steps)
    training_resume=True,                 # auto-resume if ckpt exists
)

# Sim2Real knobs
s2r_kwargs = dict(
    training_ls_coeff=1.0,                 # set to 0.0 to disable
    sim2real_real_frames=real_xyz_norm_aligned,   # must be (T,N,3) or list of (N,3)
    sim2real_train_range=(0, train_T),
    sim2real_num_pts_sim=1024,
    sim2real_num_pts_real=1024,
    sim2real_tool_t=tool_t,
    sim2real_tool_R=tool_R,
    sim2real_use_rotation=False,
    sim2real_scene_num_qp=800,
    sim2real_scene_timestep=0.01,
    sim2real_scene_newton_steps=30,
    sim2real_max_steps_per_call=50,
    sim2real_anchor_enable=True,
    sim2real_anchor_eps_ratio=0.03,
    sim2real_anchor_penalty=1e7,
    sim2real_grasp_penalty=1e5,
    
    sim2real_every=200,
    sim2real_warmup_steps=200
)

# Output dirs
run_root = os.path.join(os.getcwd(), "runs_compare")
os.makedirs(run_root, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

run_base = os.path.join(run_root, f"{stamp}_baseline_ls0")
run_s2r  = os.path.join(run_root, f"{stamp}_sim2real_ls1")

print("run_base:", run_base)
print("run_s2r :", run_s2r)

# --------------------------
# Train baseline (ls=0)
# --------------------------
sim_obj_base = ez.SimplicitsObject.create_trained(
    pts, yms, prs, rhos, approx_volume,
    training_ls_coeff=0.0,
    training_log_dir=run_base,
    **common_train_kwargs,
)

print("baseline trained:", sim_obj_base)

# --------------------------
# Train sim2real (ls>0)
# --------------------------
sim_obj_s2r = ez.SimplicitsObject.create_trained(
    pts, yms, prs, rhos, approx_volume,
    training_log_dir=run_s2r,
    **common_train_kwargs,
    **s2r_kwargs,
)

print("sim2real trained:", sim_obj_s2r)


T_total=125  train=[0,60)  eval=[60,125)
run_base: /home/nan/Desktop/kaolin/examples/tutorial/physics/runs_compare/20260318-183405_baseline_ls0
run_s2r : /home/nan/Desktop/kaolin/examples/tutorial/physics/runs_compare/20260318-183405_sim2real_ls1


  0%|          | 0/3000 [00:00<?, ?it/s]

baseline trained: <kaolin.physics.simplicits.easy_api_sim2real_UPDATED_anchor_boundary_fastckpt_v3.SimplicitsObject object at 0x7eb2217c30d0>


/home/nan/Desktop/kaolin/kaolin/physics/utils/warp_utilities.py:263: UserWarning: Sparse BSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  torch_weights = ctor(


  0%|          | 0/3000 [00:00<?, ?it/s]

sim2real trained: <kaolin.physics.simplicits.easy_api_sim2real_UPDATED_anchor_boundary_fastckpt_v3.SimplicitsObject object at 0x7eb1c3631150>


## 7) Build 1 or 2 evaluation scenes and compare side-by-side

- Choose **mode**: baseline / sim2real / both.
- Evaluation uses the same tool-driven grasp, plus **anchor boundary points** (AABB faces) to keep the bulk stable.
- Logs per-frame metrics for later plots (grasp error + optional Chamfer to real).

In [9]:

import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display
import k3d
import time, threading, os, json
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------
# Utils
# --------------------------
def torch_to_np(x): return x.detach().cpu().numpy()

@torch.no_grad()
def chamfer_l2(a: torch.Tensor, b: torch.Tensor, max_a=2048, max_b=2048):
    """
    a: (Na,3) cuda
    b: (Nb,3) cuda
    returns: scalar (mean of nearest squared distances both ways) and also mean distance (sqrt) for readability
    """
    if a.shape[0] > max_a:
        idx = torch.randint(0, a.shape[0], (max_a,), device=a.device)
        a = a[idx]
    if b.shape[0] > max_b:
        idx = torch.randint(0, b.shape[0], (max_b,), device=b.device)
        b = b[idx]
    d = torch.cdist(a, b)  # (Na,Nb)
    ab = (d.min(dim=1).values ** 2).mean()
    ba = (d.min(dim=0).values ** 2).mean()
    cd2 = ab + ba
    return cd2, torch.sqrt(cd2.clamp_min(1e-12))

def build_anchor_mask_cpu(internal_rest: torch.Tensor, eps_ratio=0.03):
    """Anchor points near AABB faces (cpu bool mask length M)."""
    mins = internal_rest.min(dim=0).values
    maxs = internal_rest.max(dim=0).values
    extent = (maxs - mins).clamp_min(1e-8)
    eps = eps_ratio * extent
    on_min_x = internal_rest[:, 0] <= (mins[0] + eps[0])
    on_max_x = internal_rest[:, 0] >= (maxs[0] - eps[0])
    on_min_y = internal_rest[:, 1] <= (mins[1] + eps[1])
    on_max_y = internal_rest[:, 1] >= (maxs[1] - eps[1])
    on_min_z = internal_rest[:, 2] <= (mins[2] + eps[2])
    on_max_z = internal_rest[:, 2] >= (maxs[2] - eps[2])
    mask_gpu = on_min_x | on_max_x | on_min_y | on_max_y | on_min_z | on_max_z
    mask_cpu = torch.zeros((internal_rest.shape[0],), dtype=torch.bool)
    mask_cpu[torch.nonzero(mask_gpu, as_tuple=False).squeeze(1).detach().cpu()] = True
    return mask_cpu

def make_scene(sim_obj, num_qp=800, newton_steps=30, timestep=0.01, direct_solve=True):
    scene = kal.physics.simplicits.SimplicitsScene()
    scene.max_newton_steps = int(newton_steps)
    scene.timestep = float(timestep)
    scene.direct_solve = bool(direct_solve)
    obj_idx = scene.add_object(sim_obj, num_qp=int(num_qp))
    scene.set_scene_gravity(acc_gravity=torch.tensor([0.0, 0.0, 0.0], device=pts.device, dtype=pts.dtype))
    scene.reset_scene()
    return scene, obj_idx

def setup_constraints(scene, obj_idx, eps_ratio=0.03, K_ANCHOR=1e7, K_GRASP=1e5, K_GRASP_PTS=10):
    """
    Build:
      - anchor mask (cpu) and anchor_target (cuda)
      - grasp ids (cuda) chosen as closest internal qp points to tool at t=0
      - grasp targets function (translation only)
    Returns a dict with all needed handles.
    """
    internal_rest = scene.get_object_deformed_pts(obj_idx)   # (M,3)
    M = internal_rest.shape[0]

    # grasp ids: closest qp to tool at t0
    t0 = tool_t[0]
    d2 = torch.sum((internal_rest - t0[None, :])**2, dim=1)
    grasp_ids = torch.topk(d2, k=int(K_GRASP_PTS), largest=False).indices

    grasp_mask_cpu = torch.zeros((M,), dtype=torch.bool)
    grasp_mask_cpu[grasp_ids.detach().cpu()] = True

    p0 = internal_rest[grasp_ids].clone()   # (K,3)
    t0c = tool_t[0].clone()

    def grasp_fcn(_): return grasp_mask_cpu

    def grasp_targets_translation(t_idx: int):
        return p0 + (tool_t[t_idx] - t0c)[None, :]

    # anchor mask: AABB faces, excluding grasp points
    anchor_mask_cpu = build_anchor_mask_cpu(internal_rest, eps_ratio=float(eps_ratio))
    anchor_mask_cpu[grasp_ids.detach().cpu()] = False

    anchor_ids_cpu = torch.nonzero(anchor_mask_cpu, as_tuple=False).squeeze(1)
    anchor_target = internal_rest[anchor_ids_cpu.to(device=internal_rest.device)].clone()

    def anchor_fcn(_): return anchor_mask_cpu

    # register boundary conditions ONCE
    scene.set_object_boundary_condition(
        obj_idx=obj_idx, name="grasp",
        fcn=grasp_fcn, bdry_penalty=float(K_GRASP),
        pinned_x=grasp_targets_translation(0),
    )
    scene.set_object_boundary_condition(
        obj_idx=obj_idx, name="anchor",
        fcn=anchor_fcn, bdry_penalty=float(K_ANCHOR),
        pinned_x=anchor_target,
    )

    return dict(
        internal_rest=internal_rest,
        grasp_ids=grasp_ids,
        grasp_ids_cpu=grasp_ids.detach().cpu(),
        grasp_targets=grasp_targets_translation,
        anchor_ids_cpu=anchor_ids_cpu,
        anchor_target=anchor_target,
        K_GRASP=float(K_GRASP),
        K_ANCHOR=float(K_ANCHOR),
    )

# --------------------------
# Choose visualization mode
# --------------------------
mode = widgets.Dropdown(
    options=[("Baseline only", "base"), ("Sim2Real only", "s2r"), ("Both (side-by-side)", "both")],
    value="both",
    description="Mode:",
)
display(mode)

# --------------------------
# Build scenes (lazy)
# --------------------------
scenes = {}

def build_if_needed():
    m = mode.value
    if m in ("base", "both") and "base" not in scenes:
        scene_b, idx_b = make_scene(sim_obj_base, num_qp=800, newton_steps=30)
        scenes["base"] = dict(scene=scene_b, obj_idx=idx_b, name="baseline")
        scenes["base"]["cons"] = setup_constraints(scene_b, idx_b, eps_ratio=0.03, K_ANCHOR=1e7, K_GRASP=1e5, K_GRASP_PTS=10)

    if m in ("s2r", "both") and "s2r" not in scenes:
        scene_s, idx_s = make_scene(sim_obj_s2r, num_qp=800, newton_steps=30)
        scenes["s2r"] = dict(scene=scene_s, obj_idx=idx_s, name="sim2real")
        scenes["s2r"]["cons"] = setup_constraints(scene_s, idx_s, eps_ratio=0.03, K_ANCHOR=1e7, K_GRASP=1e5, K_GRASP_PTS=10)

build_if_needed()

# --------------------------
# K3D visualization setup (FIXED for side-by-side)
# --------------------------
def make_plot(title, scene, obj_idx, cons):
    plot = k3d.plot(camera_auto_fit=True)

    # surface points
    surf_np = torch_to_np(scene.get_object_deformed_pts(obj_idx, orig_pts))
    surf = k3d.points(surf_np, point_size=0.01, color=0x0000ff)
    plot += surf

    # internal overlays
    internal_np = torch_to_np(scene.get_object_deformed_pts(obj_idx))
    grasp_pts = k3d.points(internal_np[cons["grasp_ids_cpu"].numpy()], point_size=0.06, color=0xff0000)
    anchor_pts = k3d.points(internal_np[cons["anchor_ids_cpu"].numpy()], point_size=0.04, color=0x00ff00)

    # tool pose + traj
    tool_pt = k3d.points(torch_to_np(tool_t[0])[None, :], point_size=0.08, color=0xffff00)
    tool_traj = k3d.line(torch_to_np(tool_t), color=0xffa500, width=0.01)

    plot += grasp_pts
    plot += anchor_pts
    plot += tool_pt
    plot += tool_traj

    # Note: k3d doesn't always render "title" reliably; keep it in UI instead
    return plot, dict(surf=surf, grasp=grasp_pts, anchor=anchor_pts, tool=tool_pt, traj=tool_traj)

plots = {}
handles = {}
plot_boxes = {}   # store Output widgets so we can update/clear them

def _show_plot_in_output(plot_obj, out_widget, header_text=""):
    out_widget.clear_output(wait=True)
    with out_widget:
        if header_text:
            print(header_text)
        plot_obj.display()  # IMPORTANT for JupyterLab/k3d reliability

def build_plots():
    global plots, handles, plot_boxes
    plots.clear(); handles.clear(); plot_boxes.clear()

    m = mode.value

    # Create output containers (these ARE ipywidgets, so HBox works)
    out_base = widgets.Output(layout={"border": "1px solid #eee", "width": "50%", "height": "520px"})
    out_s2r  = widgets.Output(layout={"border": "1px solid #eee", "width": "50%", "height": "520px"})

    if m == "base":
        pb, hb = make_plot("Baseline (ls=0)", scenes["base"]["scene"], scenes["base"]["obj_idx"], scenes["base"]["cons"])
        plots["base"], handles["base"] = pb, hb
        plot_boxes["base"] = out_base
        _show_plot_in_output(pb, out_base, header_text="Baseline (ls=0)")
        display(out_base)

    elif m == "s2r":
        ps, hs = make_plot("Sim2Real (ls>0)", scenes["s2r"]["scene"], scenes["s2r"]["obj_idx"], scenes["s2r"]["cons"])
        plots["s2r"], handles["s2r"] = ps, hs
        plot_boxes["s2r"] = out_s2r
        _show_plot_in_output(ps, out_s2r, header_text="Sim2Real (ls>0)")
        display(out_s2r)

    else:  # both
        pb, hb = make_plot("Baseline (ls=0)", scenes["base"]["scene"], scenes["base"]["obj_idx"], scenes["base"]["cons"])
        ps, hs = make_plot("Sim2Real (ls>0)", scenes["s2r"]["scene"], scenes["s2r"]["obj_idx"], scenes["s2r"]["cons"])
        plots["base"], handles["base"] = pb, hb
        plots["s2r"],  handles["s2r"]  = ps, hs
        plot_boxes["base"] = out_base
        plot_boxes["s2r"]  = out_s2r

        _show_plot_in_output(pb, out_base, header_text="Baseline (ls=0)")
        _show_plot_in_output(ps, out_s2r,  header_text="Sim2Real (ls>0)")
        display(widgets.HBox([out_base, out_s2r]))

# rebuild plots whenever mode changes (optional but nice)
def _on_mode_change(_=None):
    build_if_needed()
    build_plots()
mode.observe(_on_mode_change, names="value")

# initial build
build_plots()

def refresh_plot(key, ti):
    sc = scenes[key]["scene"]
    oi = scenes[key]["obj_idx"]
    cons = scenes[key]["cons"]
    h = handles[key]

    # surface
    h["surf"].positions = torch_to_np(sc.get_object_deformed_pts(oi, orig_pts))
    # internal overlays
    internal_np = torch_to_np(sc.get_object_deformed_pts(oi))
    h["grasp"].positions = internal_np[cons["grasp_ids_cpu"].numpy()]
    h["anchor"].positions = internal_np[cons["anchor_ids_cpu"].numpy()]
    # tool
    h["tool"].positions = torch_to_np(tool_t[ti])[None, :]

# --------------------------
# Evaluation loop with synchronized stepping + logging
# --------------------------
ti_state = {"ti": eval_T0}
running = {"flag": False}
runner = {"thread": None}

substeps = widgets.IntSlider(value=1, min=1, max=10, step=1, description="substeps")
sleep_s = widgets.FloatSlider(value=0.00, min=0.0, max=0.2, step=0.01, description="sleep(s)")
btn_next = widgets.Button(description="Next")
btn_run = widgets.Button(description="Run")
btn_stop = widgets.Button(description="Stop")
btn_reset = widgets.Button(description="Reset")

lbl = widgets.Label()
out = widgets.Output(layout={"border": "1px solid #ddd", "height": "200px", "overflow_y": "auto"})

# logs
eval_logs = []  # list of dicts (one row per step per model)
eval_out_dir = os.path.join(os.getcwd(), "eval_compare_logs")
os.makedirs(eval_out_dir, exist_ok=True)
eval_stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
eval_csv_path = os.path.join(eval_out_dir, f"eval_{eval_stamp}.csv")
eval_meta_path = os.path.join(eval_out_dir, f"eval_{eval_stamp}_meta.json")

def reset_scenes():
    for k in list(scenes.keys()):
        scenes[k]["scene"].reset_scene()
        # boundary pins are preserved in kaolin, but ensure pinned_x at t=0 is correct:
        cons = scenes[k]["cons"]
        scenes[k]["scene"].update_object_boundary_pinned_x("grasp", cons["grasp_targets"](0))
        scenes[k]["scene"].update_object_boundary_pinned_x("anchor", cons["anchor_target"])

def step_one_model(key, ti):
    sc = scenes[key]["scene"]; oi = scenes[key]["obj_idx"]; cons = scenes[key]["cons"]
    # update grasp target to current ti
    x_tgt = cons["grasp_targets"](ti)
    sc.update_object_boundary_pinned_x("grasp", x_tgt)

    # quasi-static: one Newton solve
    sc.run_sim_step()

    # metrics
    internal_now = sc.get_object_deformed_pts(oi)  # (M,3)
    grasp_err = torch.norm(internal_now[cons["grasp_ids"]] - x_tgt, dim=1).mean()

    chamfer2 = None; chamfer = None
    # optional chamfer to real if real frames available
    if (real_xyz_norm_aligned is not None) and (ti < len(real_xyz_norm_aligned)):
        real_pts = real_xyz_norm_aligned[ti]
        if isinstance(real_pts, np.ndarray):
            real_pts = torch.from_numpy(real_pts).to(device=internal_now.device, dtype=internal_now.dtype)
        else:
            real_pts = real_pts.to(device=internal_now.device, dtype=internal_now.dtype)
        chamfer2, chamfer = chamfer_l2(internal_now, real_pts, max_a=1024, max_b=1024)

    row = dict(
        split=("train" if ti < train_T else "test"),
        ti=int(ti),
        model=key,
        grasp_err=float(grasp_err.detach().cpu()),
        chamfer2=(None if chamfer2 is None else float(chamfer2.detach().cpu())),
        chamfer=(None if chamfer is None else float(chamfer.detach().cpu())),
    )
    return row

def update_all_plots(ti):
    m = mode.value
    if m in ("base", "both"):
        refresh_plot("base", ti)
    if m in ("s2r", "both"):
        refresh_plot("s2r", ti)

def do_next(_=None):
    build_if_needed()
    if not handles:
        build_plots()

    ti = ti_state["ti"]
    if ti >= eval_T1:
        lbl.value = "Done."
        return

    with out:
        print(f"\n=== ti={ti} ({'train' if ti < train_T else 'test'}) ===")

    m = mode.value
    keys = []
    if m in ("base", "both"): keys.append("base")
    if m in ("s2r", "both"): keys.append("s2r")

    for ss in range(substeps.value):
        for key in keys:
            row = step_one_model(key, ti)
            eval_logs.append(row)
            with out:
                print(f"  {key:>4}  grasp_err={row['grasp_err']:.6f}  chamfer={row['chamfer']}")
        update_all_plots(ti)
        if sleep_s.value > 0: time.sleep(sleep_s.value)

    # persist periodically
    if len(eval_logs) % 20 == 0:
        pd.DataFrame(eval_logs).to_csv(eval_csv_path, index=False)
        with open(eval_meta_path, "w") as f:
            json.dump(dict(
                train_T=int(train_T),
                eval_T0=int(eval_T0),
                eval_T1=int(eval_T1),
                mode=mode.value,
                run_base=run_base,
                run_s2r=run_s2r,
            ), f, indent=2)

    # show quick delta if both
    if mode.value == "both":
        last_base = next((r for r in reversed(eval_logs) if r["ti"]==ti and r["model"]=="base"), None)
        last_s2r  = next((r for r in reversed(eval_logs) if r["ti"]==ti and r["model"]=="s2r"), None)
        if last_base and last_s2r and (last_base["chamfer"] is not None) and (last_s2r["chamfer"] is not None):
            delta = last_base["chamfer"] - last_s2r["chamfer"]
            lbl.value = f"ti={ti:04d}/{eval_T1-1}  chamfer(base - s2r)={delta:+.6f}"
        else:
            lbl.value = f"ti={ti:04d}/{eval_T1-1}"
    else:
        lbl.value = f"ti={ti:04d}/{eval_T1-1}"

    ti_state["ti"] += 1

def run_loop():
    running["flag"] = True
    while running["flag"] and ti_state["ti"] < eval_T1:
        do_next()
    running["flag"] = False

def do_run(_=None):
    if running["flag"]:
        return
    runner["thread"] = threading.Thread(target=run_loop, daemon=True)
    runner["thread"].start()

def do_stop(_=None):
    running["flag"] = False

def do_reset(_=None):
    running["flag"] = False
    ti_state["ti"] = eval_T0
    reset_scenes()
    update_all_plots(eval_T0)
    with out:
        print("\n=== RESET ===")
    lbl.value = "Reset."

btn_next.on_click(do_next)
btn_run.on_click(do_run)
btn_stop.on_click(do_stop)
btn_reset.on_click(do_reset)

display(
    widgets.HBox([btn_next, btn_run, btn_stop, btn_reset]),
    widgets.HBox([substeps, sleep_s]),
    lbl,
    out
)

print("Eval CSV will be saved to:", eval_csv_path)


Dropdown(description='Mode:', index=2, options=(('Baseline only', 'base'), ('Sim2Real only', 's2r'), ('Both (s…

/home/nan/miniconda3/envs/subspace_mfem/lib/python3.10/site-packages/warp/_src/torch.py:280: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  if t.grad is not None:


Label(value='')

Output(layout=Layout(border='1px solid #ddd', height='200px', overflow_y='auto'))

Eval CSV will be saved to: /home/nan/Desktop/kaolin/examples/tutorial/physics/eval_compare_logs/eval_20260318-183633.csv


## 8) Plot evaluation curves and export figures

Run this after you have some evaluation steps logged.

In [10]:

import pandas as pd
import matplotlib.pyplot as plt
import os

if not os.path.exists(eval_csv_path):
    raise FileNotFoundError(f"Missing eval log CSV: {eval_csv_path}")

df = pd.read_csv(eval_csv_path)
display(df.tail())

# Split-aware plots
for metric in ["grasp_err", "chamfer"]:
    if metric not in df.columns:
        continue
    plt.figure()
    for model in sorted(df["model"].unique()):
        sdf = df[df["model"]==model]
        if metric == "chamfer" and sdf[metric].isna().all():
            continue
        plt.plot(sdf["ti"], sdf[metric], label=model)
    plt.axvline(train_T-1, linestyle="--")
    plt.title(f"{metric} vs time (dashed = train/test boundary)")
    plt.xlabel("ti")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    plt.show()

# Save a quick summary
summary = df.groupby(["model","split"]).agg(
    grasp_err_mean=("grasp_err","mean"),
    grasp_err_med=("grasp_err","median"),
    chamfer_mean=("chamfer","mean"),
    chamfer_med=("chamfer","median"),
).reset_index()
display(summary)

summary_path = eval_csv_path.replace(".csv", "_summary.csv")
summary.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)


FileNotFoundError: Missing eval log CSV: /home/nan/Desktop/kaolin/examples/tutorial/physics/eval_compare_logs/eval_20260318-183633.csv